# Phase 0 — Foundation

**Paper 1 · Unrecognized organ damage · AI-READI v3.0.0**

Phase 0 answers one question: *does the data this paper needs actually exist,
and is it usable?* Nothing is concluded here — this is the gate that decides
whether Phase 1 can start.

Four checks:

| | Question |
|---|---|
| **E0.1** | Do the three damage markers exist, and what do they look like? |
| **E0.2** | Can each marker be compared against what the participant reported? |
| **E0.3** | Will the Phase-2 extension variables have data? |
| **E0.4** | Does the assembled participant table hold up to QC? |

Narrative write-up: `reports/2026-08-11-phase0-report.md`.
Durable record: `papers/p1-unrecognized-damage/RESULTS_LOG.md`.

> **This notebook is thin by policy.** Cleaning rules, special-code handling and
> abnormality thresholds live in `src/aireadi/`. If you find yourself writing a
> transformation twice, it belongs in the package, not here.

## Setup

In [ ]:
# Thin-notebook bootstrap: put `src/` and the paper's `scripts/` on the path so
# this notebook can call the same code the command-line runners call. No
# cleaning or threshold logic is defined here -- it all lives in `src/aireadi`.
import sys, pathlib

REPO = pathlib.Path.cwd()
while not (REPO / "src" / "aireadi").exists():          # works from any cwd
    REPO = REPO.parent
sys.path[:0] = [str(REPO / "src"), str(REPO / "papers/p1-unrecognized-damage/scripts")]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from aireadi import figures as fg, results, stats, thresholds
import _phase1

fg.style()
RESULTS = results.results_dir("p1")
GROUPS = fg.SEVERITY_ORDER
pd.set_option("display.width", 180)
print("repo:", REPO.name)


### Load the master participant table

`_phase1.load()` reads the cached build of `cohort.build_p1_table()` — one row
per participant, with the damage flags applied. The table itself is
participant-level and lives in `data/processed/` (gitignored); only aggregates
computed from it ever reach `results/`.

> **Never print participant rows.** `df.head()` would put six real people into
> this notebook's saved output, and AI-READI is controlled-access under a data
> use agreement. Outputs are cleared before commit, but the safer habit is to
> inspect the *schema* rather than the rows, so an un-cleared notebook is
> harmless by construction.

In [ ]:
df = _phase1.load()
print(f"{len(df):,} participants x {df.shape[1]} columns\n")

# Schema and completeness -- aggregate only, no rows.
key = ["study_group_label", "clinical_site", "age", "hba1c", "bmi",
       "acr_mg_g", "troponin_t", "monofilament_min", "sr_kidney", "sr_heart"]
pd.DataFrame({"dtype": df[key].dtypes.astype(str),
              "n_present": df[key].notna().sum(),
              "pct_present": (df[key].notna().mean() * 100).round(1)})

## E0.1 — Do the three markers exist, and what do they look like?

The paper rests on three inexpensive tests taken at one visit. Before counting
anything, confirm each one is present, in the units we think, with sane values.

Two traps live here, both in `docs/CAVEATS.md`:

- `import_albumin` is **serum** albumin, not the kidney marker. The kidney
  marker is the **ratio** `import_urine_albumin / import_urine_creatinine`.
- 712 troponin rows carry a *detection limit*, not a measurement. A naive
  `troponin >= 6` would call almost the entire cohort abnormal.

In [ ]:
profile = pd.DataFrame({
    "marker": ["urine ACR (mg/g)", "hs-cTnT (ng/L)", "monofilament, worse foot"],
    "column": ["acr_mg_g", "troponin_t", "monofilament_min"],
    "n_measured": [df.acr_mg_g.notna().sum(), df.troponin_t.notna().sum(),
                   df.monofilament_min.notna().sum()],
}).assign(pct_of_cohort=lambda t: (t.n_measured / len(df) * 100).round(1))

profile["median"] = [round(df.acr_mg_g.median(), 2), round(df.troponin_t.median(), 2),
                     df.monofilament_min.median()]
below = int(df.troponin_t_below_detection.fillna(0).sum())
print(profile.to_string(index=False))
print(f"\ntroponin rows AT the detection limit: {below:,} "
      f"(median over detectable results only: {df.loc[~df.troponin_t_below_detection.fillna(False).astype(bool), 'troponin_t'].median():.2f})")

### Figure — the three markers as actually distributed

Worth drawing rather than tabulating: it shows *where the cutoff falls relative
to the mass of the data*. In all three, most abnormal results sit just past the
line — which is the whole argument for the E1.5 sensitivity sweep.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

ax = axes[0]
acr = df["acr_mg_g"].dropna()
ax.hist(np.log10(acr[acr > 0]), bins=44, color=fg.ORGAN["kidney"],
        edgecolor=fg.SURFACE, linewidth=0.4)
ax.axvline(np.log10(30), color=fg.INK, linewidth=1.6, linestyle="--")
ax.annotate("abnormal\n\u2265 30 mg/g", xy=(np.log10(30), ax.get_ylim()[1] * 0.82),
            xytext=(6, 0), textcoords="offset points", fontsize=8, color=fg.INK)
ax.set_xticks([0, 1, 2, 3]); ax.set_xticklabels(["1", "10", "100", "1000"])
ax.set_xlabel("urine ACR (mg/g, log scale)"); ax.set_ylabel("participants")
ax.set_title(f"Kidney \u2014 ACR   n={len(acr):,}", loc="left")

ax = axes[1]
tro = df["troponin_t"].dropna()
ax.hist(tro[tro <= 60], bins=44, color=fg.ORGAN["heart"], edgecolor=fg.SURFACE,
        linewidth=0.4)
ax.axvline(14, color=fg.INK, linewidth=1.6, linestyle="--")
ax.annotate("abnormal\n\u2265 14 ng/L", xy=(14, ax.get_ylim()[1] * 0.82),
            xytext=(6, 0), textcoords="offset points", fontsize=8, color=fg.INK)
ax.set_xlabel("hs-cTnT (ng/L), truncated at 60"); ax.set_ylabel("participants")
ax.set_title(f"Heart \u2014 troponin   n={len(tro):,}", loc="left")
ax.annotate(f"{below:,} below the 6 ng/L\ndetection limit", xy=(0.97, 0.62),
            xycoords="axes fraction", ha="right", fontsize=8, color=fg.INK_SECONDARY)

ax = axes[2]
# CAVEAT: the cutoff is defined on the WORSE FOOT (0-10 missed).
# `monofilament_insensate_sites` sums BOTH feet (0-20) and would misplace the line.
missed = (10 - df["monofilament_min"].dropna()).astype(int)
vc = missed.value_counts().sort_index()
ax.bar(vc.index, vc.values, 0.8,
       color=[fg.DEEMPHASIS if k < 2 else fg.ORGAN["nerve"] for k in vc.index],
       edgecolor=fg.SURFACE, linewidth=0.8)
ax.axvline(1.5, color=fg.INK, linewidth=1.6, linestyle="--")
ax.annotate("abnormal \u2265 2", xy=(1.5, ax.get_ylim()[1] * 0.82), xytext=(6, 0),
            textcoords="offset points", fontsize=8, color=fg.INK)
ax.set_yscale("log")
ax.set_xlabel("sites NOT felt, worse foot (of 10)"); ax.set_ylabel("participants (log)")
ax.set_title(f"Nerve \u2014 monofilament   n={len(missed):,}", loc="left")

fg.finish(fig, "E0.1 \u2014 The three markers, as actually distributed",
          "Cutoffs fixed at E1.0 shown as dashed lines. In every organ the bulk of "
          "abnormal results sits just past the line, which is why E1.5 sweeps them.",
          "Source: master participant table (AI-READI v3.0.0) \u00b7 results/E0_1_marker_profile.csv")
fig.savefig(RESULTS / "E0_1_marker_distributions.png", dpi=200, bbox_inches="tight")
fig

## E0.2 — Can each marker be compared against self-report?

"Unrecognized" is defined as *abnormal result **and** no corresponding
self-reported diagnosis*. That definition needs a self-report item per organ.

**This is where the gate triggered.** Kidney and heart map cleanly. Nerve does
not: the 30-item `mhoccur` medical-history battery contains no neuropathy,
numbness or foot item, and `condition_occurrence.csv` re-expresses the same 30
items. An exhaustive re-search at `E0.AUDIT` confirmed it — the nearest hit,
`dmlfeet`, is *"How often do you inspect your feet?"*, a self-care behaviour,
not a diagnosis.

In [ ]:
mapping = pd.DataFrame([
    {"organ": "kidney", "self_report_item": "mhoccur_rnl",
     "n_yes": int(df.sr_kidney.sum()), "usable": True},
    {"organ": "heart", "self_report_item": "mhoccur_mi | mhoccur_cvdot",
     "n_yes": int(df.sr_heart.sum()), "usable": True},
    {"organ": "nerve", "self_report_item": "NONE IN v3.0.0",
     "n_yes": 0, "usable": False},
])
print(mapping.to_string(index=False))
print("\nGATE: nerve has an excellent exam and no comparator.")
print("Resolved at E0.GATE -- nerve keeps prevalence and the multi-organ count,")
print("and is excluded from the unrecognized fraction entirely.")

## E0.4 — Does the assembled table hold up?

The QC that has to pass before any analysis: exact cohort size, exact group
sizes against the published 776 / 560 / 686 / 258, no duplicate participants,
and the documented kidney spot-check.

In [ ]:
assert len(df) == 2280, len(df)
assert not df.person_id.duplicated().any()
group_ns = df.study_group_label.value_counts().reindex(GROUPS)
print(group_ns.to_string(), "\n")
assert list(group_ns) == [776, 560, 686, 258], list(group_ns)

# The documented spot-check: how many have kidney damage, and how many of those
# reported a kidney problem?
abn = df.abn_kidney.eq(1)
print(f"ACR >= 30 mg/g:            {int(abn.sum()):>5,}")
print(f"  of whom self-report yes: {int((abn & df.sr_kidney.eq(1)).sum()):>5,}")
print(f"  of whom self-report no:  {int((abn & df.sr_kidney.eq(0)).sum()):>5,}")
print(f"  of whom refused:         {int((abn & df.sr_kidney.isna()).sum()):>5,}")
print("\nQC PASSED -- cohort identity and the spot-check both reproduce.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2),
                         gridspec_kw={"width_ratios": [1, 1.15]})

ax = axes[0]
rects = ax.bar(range(4), group_ns.values, 0.62, color=fg.SEVERITY,
               edgecolor=fg.SURFACE, linewidth=1.2)
for r, v in zip(rects, group_ns.values):
    ax.annotate(f"{v:,}", xy=(r.get_x() + r.get_width() / 2, v), xytext=(0, 3),
                textcoords="offset points", ha="center", fontsize=9,
                color=fg.INK_SECONDARY)
ax.set_xticks(range(4)); ax.set_xticklabels(GROUPS)
ax.set_ylabel("participants")
ax.set_title(f"Cohort \u2014 {len(df):,} participants", loc="left")

ax = axes[1]
cov = {"Kidney\n(ACR)": df.acr_mg_g.notna().mean() * 100,
       "Heart\n(troponin)": df.troponin_t.notna().mean() * 100,
       "Nerve\n(monofilament)": df.monofilament_min.notna().mean() * 100,
       "Kidney\nself-report": df.sr_kidney.notna().mean() * 100,
       "Heart\nself-report": df.sr_heart.notna().mean() * 100,
       "Nerve\nself-report": 0.0}
# Emphasis, not identity: five bars are fine and one is the story.
rects = ax.bar(range(6), list(cov.values()), 0.62,
               color=[fg.EMPHASIS] * 5 + ["#d03b3b"],
               edgecolor=fg.SURFACE, linewidth=1.2)
for r, v in zip(rects, cov.values()):
    ax.annotate(f"{v:.1f}%", xy=(r.get_x() + r.get_width() / 2, v), xytext=(0, 3),
                textcoords="offset points", ha="center", fontsize=8.5,
                color="#d03b3b" if v == 0 else fg.INK_SECONDARY,
                fontweight="bold" if v == 0 else "normal")
ax.set_xticks(range(6)); ax.set_xticklabels(list(cov), fontsize=8)
ax.set_xlim(-0.6, 5.9); ax.set_ylim(0, 118)
ax.set_ylabel("% of cohort with a value")
ax.set_title("Coverage \u2014 and the one gap that reshaped the paper", loc="left")
ax.annotate("NO neuropathy\nself-report item\nanywhere in v3.0.0\n(E0.GATE)",
            xy=(5, 8), ha="center", va="bottom", fontsize=7.5,
            color="#d03b3b", fontweight="bold", linespacing=1.4)

fg.finish(fig, "E0.4 \u2014 Cohort and marker coverage",
          "Group sizes match the published 776 / 560 / 686 / 258 exactly. The three "
          "markers are 97.6\u201399.5% complete; the missing piece is a comparator, not a measurement.",
          "Source: master participant table \u00b7 results/E0_4_cohort_qc_by_group.csv, E0_2_organ_self_report_map.csv")
fig.savefig(RESULTS / "E0_4_cohort_and_coverage.png", dpi=200, bbox_inches="tight")
fig

## Gate decision

**Phase 1 can start.** All three markers exist with 97.6–99.5% coverage, the
cohort reproduces exactly, and the kidney spot-check reproduces.

One scope change, agreed with Evan and logged as `E0.GATE`:

> Nerve is retained for **measured prevalence**, the **multi-organ count** and
> the Phase-2 depression aim, and is **excluded from the unrecognized
> fraction**. The broad `mhoccur_cns` / `mhoccur_circ` proxies are not used at
> all, not even as a labelled sensitivity check. The missing neuropathy item is
> stated in Limitations.

The Aim-1 unrecognized headline therefore covers **kidney and heart only** —
which is also why the working title names two organs while the results describe
three (`E1.DECIDE`).

→ Continue in `01_phase1_core_sweep.ipynb`.